In [3]:
from tqdm import tqdm

import itertools

import numpy as np
import statsmodels.api as sm


In [4]:
def mu_helper(j, delta):
    assert isinstance(delta, int) and delta >= 2, "delta must be integer larger than 2"
    
    numerator = np.sum([i - (delta-1)*0.5 for i in range(j, delta)])
    denominator = np.sum([i*(i-(delta-1)*0.5) for i in range(delta)])
    
    weight = numerator/denominator
    
    return weight

def mu(delta):
    return np.asarray([mu_helper(j, delta) for j in range(1, delta)])

In [5]:
def complicated_mu_helper(t,tminus, delta):
    num = np.sum([i-(2*t-delta+1)/2 for i in range(tminus, t+1)])
    denom = np.sum([(i-(t-delta+1))*(i-(2*t-delta+1)/2) for i in range(t-delta+1, t+1)])
    return num/denom
def complicated_mu(t, delta):
    return np.asarray([complicated_mu_helper(t, i, delta) for i in range(t-delta+2,t+1)])

In [7]:
for t in tqdm(range(10)):
    for delta in range(2,t):
        if not np.array_equal(complicated_mu(t, delta), mu(delta)):
            print("Found error in t={}, delta={}".format(t,delta))
            print((complicated_mu(t, delta), mu(delta)))

100%|██████████| 10/10 [00:00<00:00, 3476.71it/s]


### Verify working for largest $\hat{\mu}_{\lfloor \frac{t+2}{2} \rfloor -1, c'}$ when $\delta = t$

In [25]:
def mu_largest_analytical(t):
    ### Assuming delta = t
    ### Largest is in the middle int((t+2)/2)
    constant = 1.5
    
    if t % 2 == 0:
        num = t**2
    else:
        num = t**2 - 1
    den = t*(t-1)*(t+1)
    
    return constant*num/den

In [26]:
acc_mu_largest = [mu_largest_analytical(t) for t in range(2, 11)]
acc_mu_largest

[1.0,
 0.5,
 0.4,
 0.3,
 0.2571428571428571,
 0.21428571428571427,
 0.19047619047619047,
 0.16666666666666666,
 0.15151515151515152]

In [27]:
acc_mu = [mu(delta)[int((delta + 2) / 2)-1-1] for delta in range(2, 11)]
acc_mu

[1.0,
 0.5,
 0.4,
 0.3,
 0.2571428571428571,
 0.21428571428571427,
 0.19047619047619047,
 0.16666666666666666,
 0.15151515151515152]

In [31]:
diff_acc = [mu_largest_analytical(t) - mu(t)[int((t + 2) / 2)-1-1] for t in range(2, 100)]
print((np.max(diff_acc), np.min(diff_acc)))

(0.0, 0.0)


### Verify Weighted Telescopic Form is Correct

In [5]:
np.random.seed(123)  # Set the random seed to 123

# Generate random numbers
Y = np.random.rand(4)
X = np.array([0,1,2,3])


diff_Y = np.diff(Y)

X_constant = sm.add_constant(X)

In [6]:
Y, X, diff_Y

(array([0.69646919, 0.28613933, 0.22685145, 0.55131477]),
 array([0, 1, 2, 3]),
 array([-0.41032985, -0.05928788,  0.32446332]))

In [8]:
X_constant

array([[1., 0.],
       [1., 1.],
       [1., 2.],
       [1., 3.]])

In [9]:
model = sm.OLS(Y, X_constant)
results = model.fit()
results.params

array([ 0.51440636, -0.04947511])

In [10]:
beta_weighted = diff_Y@complicated_mu(X[-1],len(Y))
beta_weighted

-0.049475113093108744

In [11]:
complicated_mu(X[-1],len(Y))

array([0.3, 0.4, 0.3])

In [12]:
diff_Y

array([-0.41032985, -0.05928788,  0.32446332])

In [13]:
X@X

14

In [14]:
X@Y

2.3937865493274595

In [15]:
np.sum([(i-1.5)**2 for i in range(4)])

5.0

In [40]:
np.array([(i-1.5) for i in range(4)])@np.matrix(-np.ones(4)/4+np.diag([1 for i in range(4)]))/5

matrix([[-0.3, -0.1,  0.1,  0.3]])

In [39]:
np.matrix(-np.ones(4)/4+np.diag([1 for i in range(4)]))

matrix([[ 0.75, -0.25, -0.25, -0.25],
        [-0.25,  0.75, -0.25, -0.25],
        [-0.25, -0.25,  0.75, -0.25],
        [-0.25, -0.25, -0.25,  0.75]])

In [36]:
17/80, 73/80

(0.2125, 0.9125)

In [29]:
np.array([-1.5,-0.5,0.5,1.5])@np.array([-1.5,-0.5,0.5,1.5])

5.0

In [41]:
[(i-1.5) for i in range(4)]

[-1.5, -0.5, 0.5, 1.5]